# Claude as Judge: Evaluating LLM-Generated Summaries of arXiv Articles

This notebook uses **Claude** (Anthropic API, `claude-opus-4-20250514`) as a judge to rate LLM-generated popular-science summaries of arXiv articles.

**Evaluation criteria** (identical to `gpt_as_judge.ipynb` for cross-judge comparison):
- Style transfer intensity
- Content preservation
- Naturalness and readability

**Scoring:** Likert scale 1 (worst) to 5 (best), with a single total rating.

**Domains:** CS, Physics, Math, Economics (50 articles each, 200 total).

**Mentee:** Dorothy / Xinyue Yuan

## 1. Setup & Imports

Install the Anthropic SDK and import all dependencies.

In [ ]:
!pip install -q anthropic

In [ ]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
import re
import time
import random
import anthropic
from google.colab import userdata

# Initialize the Anthropic client with API key from Colab Secrets
claude_client = anthropic.Anthropic(api_key=userdata.get("ANTHROPIC_API_KEY"))

CLAUDE_MODEL = "claude-opus-4-20250514"
print(f"Using model: {CLAUDE_MODEL}")

## 2. Helper Functions

These functions fetch arXiv article HTML, strip boilerplate (navigation UI, table of contents), and extract the first ~4000 characters of article prose. Identical to `gpt_as_judge.ipynb`.

In [ ]:
def fetch_html_body_content(html_url):
    """Fetch an arXiv HTML page and extract the body text."""
    try:
        r = requests.get(html_url, timeout=30)
        r.raise_for_status()
        soup = BeautifulSoup(r.text, "html.parser")
    except Exception as e:
        print("Warning: HTML fetch failed:", e)
        return None, "html_fetch_failed"

    body = soup.find("body")
    if not body:
        return None, "no_body"

    lines = [line.strip() for line in body.stripped_strings if line.strip()]
    text = "\n".join(lines)

    if not text:
        return None, "empty_text"

    # Remove Abstract section
    text = re.sub(
        r'Abstract[:\s].*?(?=(Introduction|1\.\sIntroduction))',
        '',
        text,
        flags=re.S | re.I
    )

    # Truncate at Acknowledgment section
    ack_patterns = [
        r'Acknowledgment',
        r'Acknowledgements',
        r'ACKNOWLEDGMENT',
        r'ACKNOWLEDGEMENTS'
    ]
    for pat in ack_patterns:
        match = re.search(pat, text, re.I)
        if match:
            text = text[:match.start()]
            break

    text = "\n".join(line for line in text.splitlines() if line.strip())
    return text, "html_success"


# --- Boilerplate patterns to strip from arXiv HTML ---
BOILERPLATE_EXACT = {
    "Report GitHub Issue", "Submit without GitHub", "Submit in GitHub",
    "Back to arXiv", "Why HTML?", "Report Issue", "Content selection saved",
    "Describe the issue below", "×",
}

BOILERPLATE_SUBSTRINGS = [
    "Report GitHub Issue",
    "Back to arXiv",
    "Why HTML?",
    "Report Issue",
    "Submit without GitHub",
    "Submit in GitHub",
    "Back to Introduction",
    "Back to ",
    "Content selection saved",
    "Describe the issue below",
]


def get_article_snippet(full_text, max_chars=4000):
    """
    Strip the arXiv HTML preamble (nav UI + table of contents), then return
    the first max_chars of actual article prose.
    """
    lines = full_text.splitlines()

    # Phase 1: Find where the real article prose begins
    body_start = 0
    for i, line in enumerate(lines):
        stripped = line.strip()
        if not stripped:
            continue
        if len(stripped) > 80 and " " in stripped and re.search(r"[a-z]", stripped):
            body_start = i
            break

    # Phase 2: From body_start onward, drop remaining boilerplate lines
    cleaned_lines = []
    for line in lines[body_start:]:
        stripped = line.strip()
        if not stripped:
            if cleaned_lines:
                cleaned_lines.append("")
            continue
        if any(bp in stripped for bp in BOILERPLATE_SUBSTRINGS):
            continue
        if stripped in BOILERPLATE_EXACT:
            continue
        if re.match(r"^[A-Z]?\.?\d+(\.\d+)*$", stripped):
            continue
        cleaned_lines.append(line)

    cleaned = "\n".join(cleaned_lines)
    return cleaned[:max_chars]


print("Helper functions loaded.")

## 3. Claude API Call with Retry Logic

Wrapper around `client.messages.create()` with exponential backoff to handle rate limits.

In [ ]:
def call_claude_with_retry(prompt, max_retries=5):
    """
    Send a prompt to Claude and return the response text.
    Retries with exponential backoff on rate-limit or transient errors.
    """
    for attempt in range(max_retries):
        try:
            response = claude_client.messages.create(
                model=CLAUDE_MODEL,
                max_tokens=1024,
                messages=[{"role": "user", "content": prompt}],
            )
            # Extract text from the response
            return response.content[0].text
        except anthropic.RateLimitError as e:
            wait = (2 ** attempt) + random.uniform(0, 1)
            print(f"  Rate limited (attempt {attempt + 1}/{max_retries}), waiting {wait:.1f}s...")
            time.sleep(wait)
        except anthropic.APIError as e:
            wait = (2 ** attempt) + random.uniform(0, 1)
            print(f"  API error (attempt {attempt + 1}/{max_retries}): {e}, waiting {wait:.1f}s...")
            time.sleep(wait)
    raise RuntimeError(f"Failed after {max_retries} retries")


def parse_rating_response(output_text):
    """
    Extract the explanation and numeric total rating from Claude's response.
    Robust to extra text — searches all lines for the expected fields.
    """
    explanation = ""
    rating = None
    for line in output_text.splitlines():
        line_stripped = line.strip()
        if line_stripped.lower().startswith("explanation:"):
            explanation = line_stripped.split(":", 1)[1].strip()
        elif line_stripped.lower().startswith("total rating:"):
            match = re.search(r"\d+", line_stripped)
            if match:
                val = int(match.group())
                if 1 <= val <= 5:
                    rating = val
    return explanation, rating


print("API wrapper and parser loaded.")

## 4. Evaluation Prompt Template

This prompt is **identical** to the one used in `gpt_as_judge.ipynb` so that ratings from Claude and GPT are directly comparable.

In [ ]:
def build_eval_prompt(article_snippet, summary):
    """
    Construct the evaluation prompt — identical to gpt_as_judge.ipynb
    so cross-judge comparison is valid.
    """
    return f"""
Evaluate the quality of summaries written for a news article.
Rate each summary on four dimensions:
style transfer intensity, content preservation, naturalness and readability.
You should rate on a scale from 1 (worst) to 5 (best).

Article: {article_snippet}
Summary: {summary}

Provide your evaluation as follows:
Explanation: (your rationale for the rating, as a text)
Total rating: (your rating, as a number between 1 and 5)

You MUST provide values for 'Explanation:' and 'Total rating:' in your answer.
"""

## 5. Data Loading & Evaluation Loop

We process all four domains (CS, Physics, Math, Economics). Each domain has its own CSV file with 50 arXiv articles and their LLM-generated summaries.

The loop:
1. Loads the domain CSV
2. Fetches each article's HTML and extracts a prose snippet
3. Sends the snippet + summary to Claude for evaluation
4. Parses the rating and saves results to a new CSV

In [ ]:
# Domain configurations: (domain_label, input_csv, output_csv)
DOMAINS = [
    ("cs",      "results_gpt_cs.csv",      "results_gpt_cs_claude_ratings.csv"),
    ("physics", "results_gpt_phy.csv",      "results_gpt_phy_claude_ratings.csv"),
    ("math",    "results_gpt_math.csv",     "results_gpt_math_claude_ratings.csv"),
    ("econ",    "results_gpt_econ.csv",     "results_gpt_econ_claude_ratings.csv"),
]

all_results = []  # Collect DataFrames for cross-domain analysis

for domain_label, input_csv, output_csv in DOMAINS:
    print(f"\n{'='*60}")
    print(f"Processing domain: {domain_label.upper()}")
    print(f"{'='*60}")

    try:
        df = pd.read_csv(input_csv)
    except FileNotFoundError:
        print(f"  File not found: {input_csv} — skipping this domain.")
        continue

    print(f"  Loaded {len(df)} rows from {input_csv}")

    # Add columns for Claude's ratings
    df["claude_rate"] = pd.NA
    df["claude_explanation"] = ""

    for index, row in df.iterrows():
        try:
            # Fetch and clean article text
            raw_text, status = fetch_html_body_content(row["html_url"])
            if raw_text is None:
                print(f"  Row {index}: fetch failed ({status})")
                continue

            article_snippet = get_article_snippet(raw_text)
            summary = row["gpt_prompt"]

            # Build prompt and call Claude
            prompt = build_eval_prompt(article_snippet, summary)
            output_text = call_claude_with_retry(prompt)

            print(f"  Row {index}:")
            print(f"    {output_text[:200]}...")

            # Parse the response
            explanation, rating = parse_rating_response(output_text)
            df.at[index, "claude_rate"] = rating
            df.at[index, "claude_explanation"] = explanation

        except Exception as e:
            print(f"  Error at row {index}: {e}")

    # Save per-domain results
    df.to_csv(output_csv, index=False)
    print(f"  Saved {len(df)} rows to {output_csv}")

    # Tag domain for aggregation
    df["domain"] = domain_label
    all_results.append(df)

print(f"\nDone! Processed {len(all_results)} domain(s).")

## 6. Results Aggregation

Combine all domain results into a single DataFrame and compute per-domain average ratings.

In [ ]:
# Combine all domains
combined_df = pd.concat(all_results, ignore_index=True)

# Convert ratings to numeric (drop NAs for stats)
combined_df["claude_rate"] = pd.to_numeric(combined_df["claude_rate"], errors="coerce")

# Per-domain summary statistics
domain_stats = combined_df.groupby("domain")["claude_rate"].agg(
    count="count",
    mean="mean",
    std="std",
    median="median",
    min="min",
    max="max",
).round(2)

print("Per-domain Claude rating statistics:")
print(domain_stats)
print(f"\nOverall mean rating: {combined_df['claude_rate'].mean():.2f}")
print(f"Total articles rated: {combined_df['claude_rate'].notna().sum()}")

## 7. Analysis & Visualization

### 7a. Rating distribution per domain (bar chart)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# --- Bar chart: average rating by domain ---
fig, ax = plt.subplots(figsize=(8, 5))

domain_means = combined_df.groupby("domain")["claude_rate"].mean()
domain_order = ["cs", "physics", "math", "econ"]
domain_labels = ["CS", "Physics", "Math", "Economics"]

means = [domain_means.get(d, 0) for d in domain_order]
colors = ["#4C78A8", "#F58518", "#E45756", "#72B7B2"]

bars = ax.bar(domain_labels, means, color=colors, edgecolor="black", linewidth=0.5)

# Add value labels on bars
for bar, val in zip(bars, means):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.05,
            f"{val:.2f}", ha="center", va="bottom", fontweight="bold")

ax.set_ylabel("Average Claude Rating (1-5)")
ax.set_title("Claude Judge: Average Summary Rating by Domain")
ax.set_ylim(0, 5.5)
ax.axhline(y=combined_df["claude_rate"].mean(), color="gray", linestyle="--",
           label=f"Overall mean: {combined_df['claude_rate'].mean():.2f}")
ax.legend()
plt.tight_layout()
plt.show()

### 7b. Rating distribution histogram (all domains)

In [ ]:
# --- Histogram: rating distribution across all domains ---
fig, ax = plt.subplots(figsize=(8, 5))

for domain, color, label in zip(domain_order, colors, domain_labels):
    subset = combined_df[combined_df["domain"] == domain]["claude_rate"].dropna()
    ax.hist(subset, bins=[0.5, 1.5, 2.5, 3.5, 4.5, 5.5],
            alpha=0.6, label=label, color=color, edgecolor="black", linewidth=0.5)

ax.set_xlabel("Claude Rating")
ax.set_ylabel("Count")
ax.set_title("Claude Judge: Rating Distribution by Domain")
ax.set_xticks([1, 2, 3, 4, 5])
ax.legend()
plt.tight_layout()
plt.show()

### 7c. Per-domain rating counts

Quick tabular view of how many articles received each rating (1-5), broken down by domain.

In [ ]:
# Rating counts per domain
rating_counts = (
    combined_df.groupby("domain")["claude_rate"]
    .value_counts()
    .unstack(fill_value=0)
    .reindex(index=domain_order, columns=[1, 2, 3, 4, 5], fill_value=0)
)
rating_counts.index = domain_labels
rating_counts.columns = [f"Rating {c}" for c in rating_counts.columns]
print("Rating counts per domain:")
print(rating_counts)

## 8. Save Combined Results

Save the full combined DataFrame (all domains) for downstream cross-judge comparison.

In [ ]:
# Save the combined results
combined_df.to_csv("results_all_domains_claude_ratings.csv", index=False)
print(f"Saved combined results: {len(combined_df)} rows to results_all_domains_claude_ratings.csv")